# Research 59 — Purchase/collection cross-seed average

One predeclared diversity check: average the aligned seed-42 and seed-2025 purchase×collection OOF probabilities 50/50, then add the component at a fixed 20% to the reference. Compare to both individual blends on both fold partitions. No weight search or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
p_seed42 = load_probs("segment-purchase-collection-balanced-oof.csv")
p_seed2025 = load_probs("segment-purchase-collection-seed2025-balanced-oof.csv")
p_mean = 0.5 * p_seed42 + 0.5 * p_seed2025
candidates = {
    "Reference": p_base,
    "Seed-42 component, fixed 80/20": 0.8 * p_base + 0.2 * p_seed42,
    "Seed-2025 component, fixed 80/20": 0.8 * p_base + 0.2 * p_seed2025,
    "Equal cross-seed mean, fixed 80/20": 0.8 * p_base + 0.2 * p_mean,
}
rows = []
for split_seed in [42, 2025]:
    folds = list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=split_seed).split(
            np.zeros(len(y)), y
        )
    )
    for name, probs in candidates.items():
        pred = probs.argmax(axis=1)
        row = {
            "fold_partition_seed": split_seed,
            "candidate": name,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
        }
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
            base_score = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
            row[f"fold_{fold}_delta_vs_reference"] = score - base_score
        rows.append(row)
    display(pd.DataFrame(rows[-4:]).round(6))

results = pd.DataFrame(rows)
results.to_csv(ROOT / "assets/segment-purchase-collection-crossseed-screen.csv", index=False)
p_blend = candidates["Equal cross-seed mean, fixed 80/20"]
pd.DataFrame(p_blend, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-purchase-collection-crossseed-blend-oof.csv", index=False
)
print("Fixed 50/50 model average and 80/20 reference mix; no weights searched; Test not loaded.")

,fold_partition_seed,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_2_delta_vs_reference,fold_3_delta_vs_reference
0,42,Reference,0.750895,0.764683,0.000000,0.000000,0.000000
1,42,"Seed-42 component, fixed 80/20",0.751558,0.765043,0.000847,0.000533,0.000608
2,42,"Seed-2025 component, fixed 80/20",0.751408,0.764941,0.001008,0.000074,0.000455
3,42,"Equal cross-seed mean, fixed 80/20",0.751661,0.765188,0.000639,0.000438,0.001215


,fold_partition_seed,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_2_delta_vs_reference,fold_3_delta_vs_reference
0,2025,Reference,0.750895,0.764683,0.000000,0.000000,0.000000
1,2025,"Seed-42 component, fixed 80/20",0.751558,0.765043,0.000406,0.001872,-0.000289
2,2025,"Seed-2025 component, fixed 80/20",0.751408,0.764941,0.000554,0.000792,0.000192
3,2025,"Equal cross-seed mean, fixed 80/20",0.751661,0.765188,0.000812,0.001395,0.000087


Fixed 50/50 model average and 80/20 reference mix; no weights searched; Test not loaded.
